# 03. Training (Text Cell)

ADR-016/017 §3-2 model × polluter × level × dataset 학습 → metric csv.

분류 5종: LogReg+TFIDF / TextCNN / DistilBERT / BERT / RoBERTa
회귀 5종: Ridge+TFIDF / XGBoost+TFIDF / TextCNN-Reg / DistilBERT-Reg / BERT-Reg

학습 함수는 `dsc_framework.text_trainers`에서 import (검증 완료).

GPU 시간: 분류 30~50시간 + 회귀 30~50시간 = 60~100시간 (T4 가정).
Colab Pro+ 또는 Pay-as-you-go.

Output: `results/text_train_metrics.csv`


In [1]:
# ============================================================
# 0. Drive 마운트 + dsc/ 자동 검색 + sys.path 등록
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, glob, json
import numpy as np
import pandas as pd


def _find_dsc_base():
    root = '/content/drive/MyDrive'
    if not os.path.isdir(root):
        return None
    for c in [f'{root}/capstone/dsc', f'{root}/dsc', f'{root}/capstone-dsc']:
        if os.path.isfile(f'{c}/dsc_framework/__init__.py'):
            return c
    for pat in [f'{root}/*/dsc_framework/__init__.py',
                f'{root}/*/*/dsc_framework/__init__.py',
                f'{root}/*/*/*/dsc_framework/__init__.py']:
        for hit in glob.glob(pat):
            return os.path.dirname(os.path.dirname(hit))
    return None


BASE = _find_dsc_base()
if BASE is None:
    drive_root = '/content/drive/MyDrive'
    listing = os.listdir(drive_root) if os.path.isdir(drive_root) else []
    raise RuntimeError(
        'dsc_framework/ 폴더를 G드라이브에서 못 찾음.\n'
        '  1) G드라이브 클라이언트 sync 완료 확인 (commit 직후면 잠시 대기 후 재시도)\n'
        '  2) Drive 마운트 확인 — !ls /content/drive/MyDrive\n'
        f'  현재 Drive 내용: {listing[:20]}'
    )

# 누락 파일 진단 — partial sync 시 빠른 실패
REQUIRED = ['shared_metrics.py', 'classification_cell.py', 'regression_cell.py',
            'image_cell.py', 'text_cell.py', 'text_cell_regression.py',
            'text_trainers.py', 'data_type_detection.py', 'router.py',
            'text_polluters', 'image_polluters']
missing = [f for f in REQUIRED if not os.path.exists(f'{BASE}/dsc_framework/{f}')]
if missing:
    raise RuntimeError(
        f'dsc_framework/ 파일 누락: {missing}\n'
        '→ G드라이브 sync 미완료. 잠시 대기 후 재실행.\n'
        '→ Colab Drive view stale 시: drive.flush_and_unmount() 후 재마운트.'
    )

RESULTS_DIR = f'{BASE}/results'
DATA_DIR = f'{BASE}/data/text'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

if BASE not in sys.path:
    sys.path.insert(0, BASE)

print(f'BASE: {BASE}')
print(f'dsc_framework 파일: {sorted(f for f in os.listdir(f"{BASE}/dsc_framework") if not f.startswith("_"))}')


Mounted at /content/drive
BASE: /content/drive/MyDrive/capstone/dsc
dsc_framework 파일: ['classification_cell.py', 'column_detection.py', 'data_type_detection.py', 'image_cell.py', 'image_polluters', 'llm_weight_generator.py', 'prompts', 'regression_cell.py', 'router.py', 'shared_metrics.py', 'text_cell.py', 'text_cell_regression.py', 'text_polluters', 'text_trainers.py']


In [2]:
# ============================================================
# 의존성 설치 (Colab 1회 실행 후 다음 셀)
# ============================================================
%pip install -q 'transformers>=4.30' 'datasets>=2.10' 'xgboost>=1.7' 'accelerate>=1.1.0'


In [3]:
# ============================================================
# imports
# ============================================================
from dsc_framework.text_trainers import (
    CLASSIFICATION_MODELS, REGRESSION_MODELS,
)
from dsc_framework.text_polluters import (
    CompletenessTextPolluter, NoiseInjectionTextPolluter, WordShufflePolluter,
    ClassBalanceTextPolluter, LabelSwapTextPolluter,
    TargetDistributionSkewTextPolluter, TargetNoiseTextPolluter,
)
print('trainers OK:', list(CLASSIFICATION_MODELS.keys()), list(REGRESSION_MODELS.keys()))


trainers OK: ['logreg_tfidf', 'textcnn', 'distilbert', 'bert_base', 'roberta_base'] ['ridge_tfidf', 'xgb_tfidf', 'textcnn_reg', 'distilbert_reg', 'bert_base_reg']


In [4]:
# ============================================================
# 데이터셋 로드 (ADR-016 분류 3종 + ADR-017 회귀 3종)
#   - Phase 2 정식 실행 시 N_TRAIN/N_TEST를 ADR §4 sample_cap으로 키울 것
#   - sanity/dev에선 작은 sample로 시작
# ============================================================
from datasets import load_dataset
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

# 사용자가 sample size 조정. ADR §4 정식 = train 50000~200000 / test 5000~50000
N_TRAIN = 3000  # 분류·SST5는 자동 cap, 빈 dataset이면 자체 train size
N_TEST  = 500


def _slice(ds_dict, split, n, seed=42):
    ds = ds_dict[split] if split in ds_dict else ds_dict['train']
    if n is None or len(ds) <= n:
        return ds
    return ds.shuffle(seed=seed).select(range(n))


def load_all():
    """분류 3 + 회귀 3 = 6 dataset을 (tr_texts, tr_y, te_texts, te_y, task)로 반환."""
    out = {}

    # 분류
    ag = load_dataset('fancyzhx/ag_news')
    out['ag_news'] = (_slice(ag, 'train', N_TRAIN), _slice(ag, 'test', N_TEST), 'classification')

    imdb = load_dataset('stanfordnlp/imdb')
    out['imdb'] = (_slice(imdb, 'train', N_TRAIN), _slice(imdb, 'test', N_TEST), 'classification')

    news20 = load_dataset('SetFit/20_newsgroups')
    out['20news'] = (_slice(news20, 'train', N_TRAIN), _slice(news20, 'test', N_TEST), 'classification')

    # 회귀 (label = star/sentiment를 float)
    yelp = load_dataset('Yelp/yelp_review_full')
    out['yelp_full'] = (_slice(yelp, 'train', N_TRAIN), _slice(yelp, 'test', N_TEST), 'regression')

    amazon = load_dataset('SetFit/amazon_reviews_multi_en')  # ADR-017 미러
    out['amazon_en'] = (_slice(amazon, 'train', N_TRAIN), _slice(amazon, 'test', N_TEST), 'regression')

    sst = load_dataset('SetFit/sst5')
    out['sst5'] = (_slice(sst, 'train', N_TRAIN), _slice(sst, 'test', N_TEST), 'regression')

    return out


def to_lists(ds_split, task):
    texts = ds_split['text']
    labels = ds_split['label']
    if task == 'regression':
        labels = [float(y) for y in labels]
    return list(texts), list(labels)


print('load_dataset OK — load_all() 호출하면 6 dataset 로드 시작.')


device: cuda
load_dataset OK — load_all() 호출하면 6 dataset 로드 시작.


In [5]:
# ============================================================
# 학습 sweep 설정 (per-combo 체크포인트 + resume)
# ============================================================
LEVEL_GRID = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9]
SEED = 42  # 학습은 1 seed (NB_02 sweep과 메모리 절약)

POLLUTERS_CLS = {
    'completeness_text':    CompletenessTextPolluter,
    'noise_injection_text': NoiseInjectionTextPolluter,
    'word_shuffle':         WordShufflePolluter,
    'class_balance':        ClassBalanceTextPolluter,
    'label_swap':           LabelSwapTextPolluter,
}
POLLUTERS_REG = {
    'completeness_text':         CompletenessTextPolluter,
    'noise_injection_text':      NoiseInjectionTextPolluter,
    'word_shuffle':              WordShufflePolluter,
    'target_distribution_skew':  TargetDistributionSkewTextPolluter,
    'target_noise':              TargetNoiseTextPolluter,
}

# ============================================================
# Step A dev mode — 빠른 검증용 sweep 좁히기 (~2~3시간)
# 정식 풀 sweep 돌릴 땐 DEV_MODE = False 로 바꾸고 cell 6 재실행
# ============================================================
DEV_MODE = True
DEV_DATASETS = ['ag_news', 'sst5']           # task당 1개씩 (분류+회귀)
DEV_LEVELS = [0.0, 0.5, 0.9]                 # 3 levels만
DEV_MODELS_CLS = ['logreg_tfidf', 'distilbert']
DEV_MODELS_REG = ['ridge_tfidf', 'distilbert_reg']


def train_one_combo(model_name, train_fn, tr_t, tr_y, te_t, te_y, task, **kw):
    try:
        metric = train_fn(tr_t, tr_y, te_t, te_y, **kw)
        return {'metric': float(metric), 'error': None}
    except Exception as e:
        return {'metric': float('nan'), 'error': f'{type(e).__name__}: {e}'}


def _append_row(ckpt_path, row):
    """학습 1건 끝나면 즉시 디스크에 append — 화면/세션 끊겨도 살림."""
    header = not os.path.exists(ckpt_path)
    pd.DataFrame([row]).to_csv(ckpt_path, mode='a', header=header, index=False)


def model_sweep(name, tr_ds, te_ds, task, ckpt_path, done,
                levels=None, models_filter=None):
    """done: set of (dataset, polluter, level, model) — 이미 끝난 거 skip.

    levels: None이면 LEVEL_GRID 전체 사용. list 주면 그것만.
    models_filter: None이면 task별 전체 모델. list 주면 그 이름들만.
    """
    tr_texts, tr_y = to_lists(tr_ds, task)
    te_texts, te_y = to_lists(te_ds, task)
    polluters = POLLUTERS_CLS if task == 'classification' else POLLUTERS_REG
    all_models = CLASSIFICATION_MODELS if task == 'classification' else REGRESSION_MODELS
    models = {k: v for k, v in all_models.items()
              if models_filter is None or k in models_filter}
    use_levels = levels if levels is not None else LEVEL_GRID
    new_rows = 0
    skipped = 0
    for pol_name, pol_cls in polluters.items():
        for lvl in use_levels:
            lvl_f = float(lvl)
            # (pol, lvl)의 모든 model이 done이면 pollution 자체 skip (비용 절약)
            pending = [m for m in models if (name, pol_name, lvl_f, m) not in done]
            if not pending:
                skipped += len(models)
                continue
            pol = pol_cls(lvl, random_seed=SEED)
            tr_p, tr_yp = pol.pollute(tr_texts, tr_y)
            for model_name, train_fn in models.items():
                key = (name, pol_name, lvl_f, model_name)
                if key in done:
                    skipped += 1
                    continue
                t0 = __import__('time').time()
                res = train_one_combo(model_name, train_fn, tr_p, tr_yp,
                                       te_texts, te_y, task)
                row = {
                    'dataset': name, 'task': task, 'model': model_name,
                    'polluter': pol_name, 'level': lvl_f, 'seed': SEED,
                    'metric': res['metric'], 'error': res['error'],
                    'elapsed_s': round(__import__('time').time() - t0, 2),
                }
                _append_row(ckpt_path, row)  # per-combo 체크포인트
                done.add(key)
                new_rows += 1
                print(f'  {model_name:14s} pol={pol_name:24s} lvl={lvl:.2f} metric={res["metric"]:.3f}')
    return new_rows, skipped


In [6]:
# ============================================================
# 전체 학습 sweep — long-running. Per-combo 체크포인트 + resume.
# DEV_MODE=True  : ~60 학습 (2~3시간). dev 검증용. ckpt = text_train_metrics_dev.csv
# DEV_MODE=False : 900 학습 (60~100시간). 정식. ckpt = text_train_metrics.csv
# 세션 끊겨도 같은 셀 재실행 시 끝난 거 자동 skip.
# ============================================================
import time
datasets = load_all()

if DEV_MODE:
    datasets = {k: v for k, v in datasets.items() if k in DEV_DATASETS}
    levels_use = DEV_LEVELS
    models_cls = DEV_MODELS_CLS
    models_reg = DEV_MODELS_REG
    ckpt = f'{RESULTS_DIR}/text_train_metrics_dev.csv'
    print(f'[DEV_MODE] datasets={list(datasets.keys())} levels={levels_use}')
    print(f'           models_cls={models_cls} models_reg={models_reg}')
else:
    levels_use = None         # 전체 LEVEL_GRID
    models_cls = None         # 전체 CLASSIFICATION_MODELS
    models_reg = None         # 전체 REGRESSION_MODELS
    ckpt = f'{RESULTS_DIR}/text_train_metrics.csv'
    print('[FULL] 정식 sweep — 60~100시간 예상')

# Resume: 기존 ckpt에서 done set 로드
done = set()
if os.path.exists(ckpt):
    prev = pd.read_csv(ckpt)
    prev = prev.drop_duplicates(
        subset=['dataset', 'polluter', 'level', 'model', 'seed'], keep='last'
    )
    prev.to_csv(ckpt, index=False)
    done = set(zip(prev.dataset, prev.polluter,
                   prev.level.astype(float), prev.model))
    print(f'[resume] {len(done)} combos 이미 완료 — skip 처리')
else:
    print('[fresh] 체크포인트 없음 — 처음부터 시작')

t_total = time.time()
for name, (tr_ds, te_ds, task) in datasets.items():
    print()
    print(f'=== {name} ({task}) 학습 시작 ===')
    t0 = time.time()
    mf = models_cls if task == 'classification' else models_reg
    new_rows, skipped = model_sweep(
        name, tr_ds, te_ds, task, ckpt, done,
        levels=levels_use, models_filter=mf,
    )
    print(f'  new={new_rows} skipped={skipped} ({(time.time()-t0)/60:.1f}분)')

# 최종 정리 — 중복 제거 후 다시 저장
final = pd.read_csv(ckpt)
final = final.drop_duplicates(
    subset=['dataset', 'polluter', 'level', 'model', 'seed'], keep='last'
)
final.to_csv(ckpt, index=False)
print()
print(f'[done] {ckpt} ({len(final)} rows, total {(time.time()-t_total)/60:.1f}분)')
final.head()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Step,Training Loss


  distilbert_reg pol=target_noise             lvl=0.90 metric=0.467
  new=30 skipped=0 (12.9분)

[done] /content/drive/MyDrive/capstone/dsc/results/text_train_metrics_dev.csv (60 rows, total 37.4분)


,dataset,task,model,polluter,level,seed,metric,error,elapsed_s
0,ag_news,classification,logreg_tfidf,completeness_text,0.0,42,0.814,NaN,2.36
1,ag_news,classification,distilbert,completeness_text,0.0,42,0.888,NaN,103.27
2,ag_news,classification,logreg_tfidf,completeness_text,0.5,42,0.786,NaN,1.51
3,ag_news,classification,distilbert,completeness_text,0.5,42,0.880,NaN,81.13
4,ag_news,classification,logreg_tfidf,completeness_text,0.9,42,0.250,NaN,0.29


---

다음: `04_scoreboard_text.ipynb` — r·hold-out·default vs tuned.
